Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision import datasets, transforms, models

from helper import make_dataset, train, test, get_indices
import matplotlib.pyplot as plt
import numpy as np

Use device

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Fill in respective folder location for dataset

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),  
    transforms.RandomRotation(5),      
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_real_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\train\real"
train_fake_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\train\fake"
valid_real_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\valid\real"
valid_fake_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\valid\fake"

train_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\train"
valid_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\valid"

Create train, valid and test datasets and dataloaders

In [ ]:

train_dataset = make_dataset(train_dir, 9500, 500, transform)

valid_dataset = make_dataset(valid_dir, 950, 50, transform)

val_size = int(0.5 * len(valid_dataset))
test_size = len(valid_dataset) - val_size

val_dataset, test_dataset = random_split(valid_dataset, [val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

real_count = 0
fake_count = 0

for _, labels in train_loader:
    real_count += (labels == 0).sum().item()
    fake_count += (labels == 1).sum().item()

print("Train real:", real_count)
print("Train fake:", fake_count)



ResNet18

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Sequential(
    nn.Linear(512, 128),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(128, 32),
    nn.ReLU(inplace=True),
    nn.Dropout(0.4),
    nn.Linear(32, 2),
)
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.layer3.parameters():
    param.requires_grad = True

In [ ]:
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0,9.0]).to(device))
optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 1e-5, 'weight_decay': 1e-4},
    {'params': model.layer4.parameters(), 'lr': 1e-5, 'weight_decay': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-4, 'weight_decay': 1e-4}
])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)
loss_values, val_accuracies, best_thereshold = train(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=20, scheduler=scheduler)
test(model, test_loader, device, threshold=best_thereshold)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(loss_values, label='Train Loss')
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

# Accuracy curve
ax2.plot(val_accuracies, label='Validation Accuracy', color='orange')
ax2.set_title('Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy %')
ax2.legend()

plt.tight_layout()
plt.savefig('training_curves.png')  # saves to file
plt.show()